In [0]:
# ── Gold: top_transfers ──────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql import Window

# Read from bronze transfers and active silver players
transfers = spark.table("football_catalog.bronze.transfers")
players = spark.table("football_catalog.silver.dim_players").filter("is_current = True").select("player_id", "player_name")

print("--- Processing: top_transfers ---")

# Clean the transfers data and cast the fee (using the correct column name!)
transfers_clean = (transfers
    .filter(F.col("transfer_fee").isNotNull()) 
    .select(
        F.col("player_id").cast("integer"),
        F.col("transfer_season").alias("season"),
        F.col("transfer_date").cast("date"),
        F.col("from_club_name"),
        F.col("to_club_name"),
        F.col("transfer_fee").cast("double").alias("transfer_fee_eur")
    )
)

# Join with players to get the clean name
transfers_with_names = transfers_clean.join(players, "player_id", "inner")

# Rank the top transfers globally per season
transfer_window = Window.partitionBy("season").orderBy(F.desc("transfer_fee_eur"))

gold_transfers = (transfers_with_names
    .withColumn("season_rank", F.rank().over(transfer_window))
    # Filter to only keep the top 100 transfers per season to keep the Gold table lightweight
    .filter(F.col("season_rank") <= 100) 
    .select(
        "season_rank", "season", "transfer_date", "player_name", 
        "from_club_name", "to_club_name", "transfer_fee_eur"
    )
    .orderBy(F.desc("season"), F.asc("season_rank"))
)

(gold_transfers.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("football_catalog.gold.top_transfers"))

print("SUCCESS: top_transfers created")